# Introducció a les corbes el·líptiques amb Sage

En aquest quadern mostram una breu introducció a les corbes el·líptiques, les corbes de Montgomery, i les isogènies amb Sage.


## Corbes El·líptiques definides sobre $\mathbb{F}_p$

Tractarem amb corbes el·líptiques definides sobre $\mathbb{F}_p$, amb $p$ un nombre primer.

Fixarem $p=83$ i considerarem la corba el·líptica $E_0\colon y^2=x^3+x$ definida sobre $\mathbb{F}_p$. En Sage, podem emprar la funció `EllipticCurve`, que ens retorna una corba el·líptica a partir dels coeficients de les equacions de Weierstrass (concretament, podem passar per paràmetre tots els coeficients $A_1,A_2,A_3,A_4,A_6$, o simplement $A_4$ i $A_6$ en cas que considerem l'equació de Weierstrass per característica diferent de 2 o 3). A més, mostrarem com obtenir l'invariant $j$ i també si la corba és ordinària o supersingular.

In [1]:
# Fixam paràmetres i E0
p = 83
Fp = GF(p)
E0 = EllipticCurve(Fp, [1, 0])  # y^2 = x^3 + x

# Mostram la informació bàsica de la corba
print("p:             ", p)
print("Fp:            ", Fp)
print("E_0:           ", E0)
print("j:             ", E0.j_invariant())
print("Supersingular: ", E0.is_supersingular())

p:              83
Fp:             Finite Field of size 83
E_0:            Elliptic Curve defined by y^2 = x^3 + x over Finite Field of size 83
j:              68
Supersingular:  True


Podem recórrer tots els punts de la EC amb un senzill bucle. En aquest exemple, mostram tots els punts $\mathbb{F}_p$\-racionals de $E_0$, a més de donar el total de punts i l'ordre de cada punt.



In [2]:
punts = [P for P in E0]  # Només ens dona els punts que són al cos de definició (Fp)
print("#E(Fp) =", len(punts))  # Mostram el total de punts Fp-racionals
print()

# Mostram les coordenades de cada punt i el seu ordre
for P in punts:
    print(P, f"(ordre {P.order()})")

#E(Fp) = 84

(0 : 1 : 0) (ordre 1)
(0 : 0 : 1) (ordre 2)
(2 : 33 : 1) (ordre 28)
(2 : 50 : 1) (ordre 28)
(3 : 14 : 1) (ordre 21)
(3 : 69 : 1) (ordre 21)
(4 : 20 : 1) (ordre 21)
(4 : 63 : 1) (ordre 21)
(15 : 30 : 1) (ordre 28)
(15 : 53 : 1) (ordre 28)
(17 : 38 : 1) (ordre 21)
(17 : 45 : 1) (ordre 21)
(18 : 17 : 1) (ordre 84)
(18 : 66 : 1) (ordre 84)
(21 : 22 : 1) (ordre 42)
(21 : 61 : 1) (ordre 42)
(24 : 30 : 1) (ordre 84)
(24 : 53 : 1) (ordre 84)
(28 : 20 : 1) (ordre 42)
(28 : 63 : 1) (ordre 42)
(29 : 4 : 1) (ordre 7)
(29 : 79 : 1) (ordre 7)
(31 : 5 : 1) (ordre 42)
(31 : 78 : 1) (ordre 42)
(33 : 23 : 1) (ordre 42)
(33 : 60 : 1) (ordre 42)
(42 : 29 : 1) (ordre 28)
(42 : 54 : 1) (ordre 28)
(43 : 6 : 1) (ordre 28)
(43 : 77 : 1) (ordre 28)
(44 : 30 : 1) (ordre 42)
(44 : 53 : 1) (ordre 42)
(45 : 6 : 1) (ordre 84)
(45 : 77 : 1) (ordre 84)
(46 : 40 : 1) (ordre 84)
(46 : 43 : 1) (ordre 84)
(47 : 28 : 1) (ordre 12)
(47 : 55 : 1) (ordre 12)
(48 : 1 : 1) (ordre 3)
(48 : 82 : 1) (ordre 3)
(49 : 2 

Notem que, per la Proposició 21 de la memòria, ja sabíem que $\#E(\mathbb{F}_p)=84=p+1$. 

Observem que Sage ens dona les coordenades projectives dels punts. Ara bé, excepte pel punt de l'infinit, les coordenades tenen la forma $[x:y:1]$ i, per tant, les coordenades afins corresponents són senzillament $(x,y)$.

Ara, agafarem dos punts $P,Q\in E(\mathbb{F}_p)$ per realitzar algunes operacions sobre ells.



In [3]:
P = E0(48, 1) # El podem donar en coordenades afins
Q = E0(2, 50)

# Mostram els punts
print(f"P = {P}, d'ordre {P.order()}")
print(f"Q = {Q}, d'ordre {Q.order()}")
print()

# Fem algunes operacions
print("[2]P      =", 2 * P)
print("[3]P      =", 3 * P)
print("-Q        =", -Q)
print("P + Q     =", P + Q)
print("[2]P - Q  =", 2 * P - Q)

P = (48 : 1 : 1), d'ordre 3
Q = (2 : 50 : 1), d'ordre 28

[2]P      = (48 : 82 : 1)
[3]P      = (0 : 1 : 0)
-Q        = (2 : 33 : 1)
P + Q     = (45 : 77 : 1)
[2]P - Q  = (45 : 6 : 1)


Com podem veure, a Sage és molt còmode realitzar operacions amb els punts d'una EC.

## Corbes de Montgomery amb $B=1$



Recordem que una corba de Montgomery definida sobre $K$ ve donada per l'equació $B y^2 = x^3 + A x^2 + x$, per alguns $A,B\in K$ amb $B(A^2-4)\neq 0$. En aquest treball tractam sobretot amb corbes de Montgomery amb $B=1$. A continuació, donam els mètodes per treballar amb elles a Sage.

En concret, donam un mètode per construir corbes de Montgomery amb $B=1$ i un altre mètode per obtenir el coeficient de Montgomery a partir d'una EC (donada pel model de Weierstrass). Aquests mètodes s'han agafat del script `supersingular.sage` de la implementació original de CSIDH: https://yx7.cc/code/csidh/csidh-20180427.tar.xz. No implementarem l'aritmètica de Montgomery, és a dir, Sage treballarà sobre el model de Weierstrass encara que tractem amb corbes de Montgomery. Ho fem així per simplificar, però en una implementació real de CSIDH sí que s'aconsella treballar completament amb corbes de Montgomery.


In [4]:
# A partir de A, retorna E_A: y**2 = x**3 + A x**2 + x
def montgomery_curve(A):
    return EllipticCurve(Fp, [0, A, 0, 1, 0])

# A partir de E, obté el coeficient de Mongomery A
# (En cas que E sigui una corba de Montgomery)
def montgomery_coefficient(E):
    Ew = E.change_ring(GF(p)).short_weierstrass_model()
    _, _, _, a, b = Ew.a_invariants()
    R.<z> = Fp[]
    r = (z**3 + a*z + b).roots(multiplicities=False)[0]
    s = sqrt(3 * r**2 + a)
    if not is_square(s): s = -s
    A = 3 * r / s
    assert montgomery_curve(A).change_ring(Fp).is_isomorphic(Ew)
    return Fp(A)

Per exemple, comprovem que el coeficient de Montgomery de $E_0$ és 0.  



In [5]:
montgomery_coefficient(E0)

0

També, podem obtenir els nombres $A\in\mathbb{F}_p\setminus\{\pm 2\}$ tals que $E_A$ és supersingular:



In [6]:
supersingulars = []
for A in range(p):  # Ho comprovam per cada A (excepte 2 i -2)
    if A != 2 and A != p - 2:
        if montgomery_curve(A).is_supersingular():
            supersingulars.append(A)
            
print("Corbes supersingulars:", supersingulars)


Corbes supersingulars: [0, 6, 11, 12, 13, 70, 71, 72, 77]


Aleshores, podem agafar per exemple $A=6$ per construir la corba $E_6$ i, llavors, obtenir el seu model de Weierstrass simplificat i comprovar que el seu coeficient de Montgomery és 6:



In [7]:
E6 = montgomery_curve(6); E6

Elliptic Curve defined by y^2 = x^3 + 6*x^2 + x over Finite Field of size 83

In [8]:
E6.short_weierstrass_model()

Elliptic Curve defined by y^2 = x^3 + 20*x + 57 over Finite Field of size 83

In [9]:
montgomery_coefficient(E6)

6

Endemés, Sage també proporciona un mètode per calcular el twist quadràtic. Per exemple, comprovem que $E_6^t=E_{-6}=E_{77}$:

In [10]:
# Twist quadràtic amb d = -1
E6t = E6.quadratic_twist(-1)

print(f"E_6^t =", E6t)
print(f"      = E_{montgomery_coefficient(E6t)}")

E_6^t = Elliptic Curve defined by y^2 = x^3 + 59*x^2 + 16*x over Finite Field of size 83
      = E_77


## Isogènies

Recordem que donada $E$ una EC i $S$ un subgrup finit de $E$, existeix una única EC $E/S$ i una única isogènia \(separable\) $\varphi_S : E \longrightarrow E/S$ tal que $\ker \varphi_S = S$, mòdul isomorfisme (Proposició 7 de la memòria).  


Aquesta isogènia es pot calcular explícitament amb les fórmules de Vélu. A Sage, es pot emprar la funció `isogeny()` des de la corba el·líptica de partida i indicar-hi com argument el conjunt de punts generadors del subgrup.  

Per exemple, considerarem la isogèna que parteix de $E_0$ i amb nucli $\langle P \rangle$, on $P=(48,1)$ és el punt considerat anteriorment.




In [11]:
phi = E0.isogeny(P)  # Obtenim la isogènia amb nucli <P>

# Mostram el codomini, l'expressió algebraica, i el grau
print(f"phi:   E_0 ->", phi.codomain())
print(f"phi: (x,y) ->", phi.rational_maps())
print(f"deg(phi) =", phi.degree())  # deg(phi) = #<P> = 3
print()

# Mostram la imatge dels punts P i Q
print("phi(P) =", phi(P))  # Hauria de ser [0:1:0]
print("phi(Q) =", phi(Q))  # Hauria de ser diferent a [0:1:0]
print()

# Calculam la isogènia dual i obtenim la seva expressió algebraica
print("phi_dual:", phi.dual())
print("phi_dual: (x,y) ->", phi.dual().rational_maps())

phi:   E_0 -> Elliptic Curve defined by y^2 = x^3 + 10*x + 29 over Finite Field of size 83
phi: (x,y) -> ((x^3 - 13*x^2 + 28*x + 24)/(x^2 - 13*x - 20), (x^3*y + 22*x^2*y - 25*x*y + 19*y)/(x^3 + 22*x^2 + 23*x - 36))
deg(phi) = 3

phi(P) = (0 : 1 : 0)
phi(Q) = (11 : 15 : 1)

phi_dual: Isogeny of degree 3 from Elliptic Curve defined by y^2 = x^3 + 10*x + 29 over Finite Field of size 83 to Elliptic Curve defined by y^2 = x^3 + x over Finite Field of size 83
phi_dual: (x,y) -> ((37*x^3 + 32*x^2 + 19*x + 38)/(x^2 + 39*x - 14), (40*x^3*y + 16*x^2*y - 33*x*y + 29*y)/(x^3 + 17*x^2 + 41*x - 24))


Per tant, amb Sage podem calcular isogènies de manera molt senzilla, i podem obtenir fàcilment la seva expressió algebraica, el codomini, el grau, la isogènia dual, i també podem avaluar la isogènia a diferents punts.